# Laboratorio MongoDB Replica Set

Evidencias de la Tarea 3 - INF-239. Se documenta el despliegue de Docker, la ingesta del corpus de discursos y las consultas semanticas.

## Pasos cubiertos
- Despliegue del Replica Set en Docker Compose.
- Ingesta del corpus `DiscursosOriginales` calculando hashes SHA-256 y embeddings Sentence-Transformers.
- Script de consultas semanticas (top 5 por similitud coseno).
- Guion para probar consistencia y alta disponibilidad.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess

PROJECT_ROOT = Path('.').resolve()
DATASET_DIR = PROJECT_ROOT / 'DiscursosOriginales'
EXPORT_PATH = PROJECT_ROOT / 'discursos-final-rol1-rol2.json'
CONNECTION_STRING = 'mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replicaset&readPreference=primary&appname=MongoDB%20Compass&ssl=false'

print(f'Proyecto: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_DIR} | Archivos: {len(list(DATASET_DIR.glob("*.txt")))}')
print(f'Ultima actualizacion del JSON final: {datetime.fromtimestamp(EXPORT_PATH.stat().st_mtime)}')

def run_cmd(cmd, check=False):
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(proc.stderr)
    return proc


Proyecto: C:\Users\sofia\OneDrive\Escritorio\BDD\Tarea 3\Tarea 3\tarea3-bd
Dataset: C:\Users\sofia\OneDrive\Escritorio\BDD\Tarea 3\Tarea 3\tarea3-bd\DiscursosOriginales | Archivos: 680
Ultima actualizacion del JSON final: 2025-11-06 19:26:37.520532


In [2]:
# 1) Despliegue de los contenedores (requiere Docker Desktop corriendo)
result = run_cmd(['docker', 'compose', '-f', 'deploy-202373011-4-202304074-6.yml', 'up', '-d'])
if result.returncode != 0:
    print('[WARN] Inicia Docker Desktop y vuelve a ejecutar la celda.')


$ docker compose -f deploy-202373011-4-202304074-6.yml up -d


time="2025-11-06T19:33:52-03:00" level=warning msg="C:\\Users\\sofia\\OneDrive\\Escritorio\\BDD\\Tarea 3\\Tarea 3\\tarea3-bd\\deploy-202373011-4-202304074-6.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
unable to get image 'mongo:7.0': error during connect: Get "http://%2F%2F.%2Fpipe%2FdockerDesktopLinuxEngine/v1.51/images/mongo:7.0/json": open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.
[WARN] Inicia Docker Desktop y vuelve a ejecutar la celda.


In [3]:
# 2) Verificar el estado del Replica Set
from pymongo import MongoClient
try:
    client = MongoClient(CONNECTION_STRING, serverSelectionTimeoutMS=2000)
    status = client.admin.command('hello')
    print(f"Replica Set: {status.get('setName')} | Primario: {status.get('primary')}")
    print('Miembros:', [m['name'] for m in status.get('hosts', [])])
except Exception as exc:
    print('No fue posible consultar el Replica Set. Mensaje recibido:')
    print(exc)
    print('Ejecuta la celda nuevamente cuando los contenedores esten activos.')


No fue posible consultar el Replica Set. Mensaje recibido:
No replica set members found yet, Timeout: 2.0s, Topology Description: <TopologyDescription id: 690d2250b6bc5889d96a9733, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo1', 30001) server_type: Unknown, rtt: None>, <ServerDescription ('mongo2', 30002) server_type: Unknown, rtt: None>, <ServerDescription ('mongo3', 30003) server_type: Unknown, rtt: None>]>
Ejecuta la celda nuevamente cuando los contenedores esten activos.


In [4]:
# 3) Ingesta del corpus (quitar --skip-db cuando Mongo este arriba)
ingest = run_cmd(['python', 'conectarMongo.py', '--ingest', '--skip-db'])


$ python conectarMongo.py --ingest --skip-db


Archivo JSON generado en discursos-final-rol1-rol2.json
Batches: 100%|##########| 43/43 [00:12<00:00,  3.39it/s]


In [5]:
# 4) Inspeccion del JSON final generado
with EXPORT_PATH.open('r', encoding='utf-8') as handler:
    docs = json.load(handler)

print(f'Se procesaron {len(docs)} discursos.')
print('Ejemplo de documento:')
example = docs[0]
print({'_id': example['_id'], 'archivo': example['metadata']['filename']})
print('Primeros 180 caracteres del texto original:')
print(example['texto'][:180].replace('\n', ' ') + '...')


Se procesaron 680 discursos.
Ejemplo de documento:
{'_id': '44b5a3c1bb8f59b2593341223422c48ed4f78fb31ff70a0781751434b44419cd', 'archivo': '100020.txt'}
Primeros 180 caracteres del texto original:
Muy buenos días:     Disfruten de este Sol de invierno. Gabriela Mistral decía que “hay que amar a nuestras ciudades” y agregaba que la belleza de nuestras ciudades nos ennoblece y...


In [6]:
# 5) Consulta semantica offline (usa el JSON exportado para mostrar resultados)
import numpy as np
from sentence_transformers import SentenceTransformer

query = 'Educacion publica y crecimiento economico'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
query_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
emb_matrix = np.array([doc['embedding'] for doc in docs], dtype=np.float32)
scores = emb_matrix @ query_emb
ordered = np.argsort(-scores)[:5]
for position, idx in enumerate(ordered, start=1):
    doc = docs[idx]
    snippet = doc['texto'][:160].replace('\n', ' ')
    print(f"[{position}] {doc['metadata']['filename']} | score={scores[idx]:.4f}")
    print(snippet + '...')
    print('-' * 70)


C:\Users\sofia\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[1] 82157.txt | score=0.6314
Muy buenos días:     Quiero saludar con mucho cariño a la ministra de Educación, al presidente de la FIDE, a los directivos, pero principalmente a todos ustedes...
----------------------------------------------------------------------
[2] 80522.txt | score=0.5985
Muy buenos días:     La verdad es que aquí se ve el espíritu de innovación, de emprendimiento, cómo ha cambiado este barrio, cómo está mejorando la calidad de n...
----------------------------------------------------------------------
[3] 97967.txt | score=0.5660
Muchas gracias Primer Ministro Abe, por esta cálida bienvenida.     Tanto la teoría económica como la historia económica nos demuestran en forma elocuente que h...
----------------------------------------------------------------------
[4] 86296.txt | score=0.5638
Muy buenos días:     Hoy hemos recibido, de parte de este grupo de trabajo, presidido por la ex ministra Mariana Aylwin, este Informe “Todos al Aula”, que es un...
---------------

## Pruebas de robustez
1. Ejecutar `docker compose -f deploy-202373011-4-202304074-6.yml up -d` y esperar a que `rs-init` finalice.
2. Ingresar registros desde el primario (`python conectarMongo.py --ingest`) y confirmar en secundarios usando `rs.status()` y `db.Discursos.countDocuments()`.
3. Alta disponibilidad: `docker stop mongo1` y verificar que `mongo2` o `mongo3` asumen el rol de PRIMARY.
4. Restaurar `mongo1` con `docker start mongo1` y revisar que el cluster vuelva al estado estable.
Documentar las salidas en esta notebook o adjuntar capturas durante la defensa.

In [7]:
# 6) Limpieza opcional de la infraestructura
cleanup = run_cmd(['docker', 'compose', '-f', 'deploy-202373011-4-202304074-6.yml', 'down'])
if cleanup.returncode != 0:
    print('Si los contenedores no estaban arriba, este mensaje se puede ignorar.')


$ docker compose -f deploy-202373011-4-202304074-6.yml down


time="2025-11-06T19:34:23-03:00" level=warning msg="C:\\Users\\sofia\\OneDrive\\Escritorio\\BDD\\Tarea 3\\Tarea 3\\tarea3-bd\\deploy-202373011-4-202304074-6.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
error during connect: Get "http://%2F%2F.%2Fpipe%2FdockerDesktopLinuxEngine/v1.51/containers/json?all=1&filters=%7B%22label%22%3A%7B%22com.docker.compose.config-hash%22%3Atrue%2C%22com.docker.compose.oneoff%3DFalse%22%3Atrue%2C%22com.docker.compose.project%3Dtarea3-bd%22%3Atrue%7D%7D": open //./pipe/dockerDesktopLinuxEngine: The system cannot find the file specified.
Si los contenedores no estaban arriba, este mensaje se puede ignorar.
